In [13]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
  
from sklearn.preprocessing import PowerTransformer

In [15]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [17]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [19]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

## Test dataset: MAASTRO 

In [21]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [23]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [25]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [27]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [29]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [31]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [33]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# Set y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [35]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [37]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [39]:
# Change the name of a column 'OS_event' in the clincial_test 
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [41]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

## Feature Selection

In [43]:
# Choose features from the result of Cox PLSR in R
selectd_features = [
"first_order_Maximum_CT",
"LBP_201_PET",
"LBP_102_PET",
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16",
"LBP_003_PET",
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2"    
] 

In [45]:
X_plsr = X.loc[:, selectd_features]
X_new = X_plsr.copy()

In [47]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selectd_features]

# Yeo-Johnson Transformation

In [49]:
# Copy the original X for later 
original_X = X.copy()

In [51]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)

# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [53]:
X_new

,first_order_Maximum_CT,LBP_201_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,LBP_003_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,1763.283818,0.000062,0.000000,0.000416,0.000123,0.000959
1,1432.109636,0.000349,0.000000,0.001753,0.000349,0.002776
2,1685.931373,0.000000,0.000034,0.000230,0.000000,0.001179
3,1329.347515,0.000000,0.000000,0.000711,0.000300,0.002748
4,1197.225910,0.000399,0.000199,0.001033,0.000000,0.002309
...,...,...,...,...,...,...
134,1267.959716,0.000000,0.000000,0.000226,0.000000,0.001347
135,1788.278096,0.000079,0.000000,0.000183,0.000039,0.000850
136,1478.297861,0.000000,0.000000,0.000621,0.000000,0.001111
137,1207.068493,0.000054,0.000000,0.000265,0.000000,0.000935


In [55]:
X_new_std

,first_order_Maximum_CT,LBP_201_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,LBP_003_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,1.022943,-0.074611,-0.784694,-0.128003,1.042615,-0.456825
1,0.072727,1.689445,-0.784694,1.761655,1.929439,1.829690
2,0.853782,-1.065926,0.236612,-0.861897,-0.799521,0.033412
3,-0.390532,-1.065926,-0.784694,0.668885,1.846691,1.814149
4,-1.187658,1.782787,1.784568,1.208077,-0.799521,1.522029
...,...,...,...,...,...,...
134,-0.727488,-1.065926,-0.784694,-0.877225,-0.799521,0.352487
135,1.072604,0.138716,-0.784694,-1.082845,-0.000653,-0.735470
136,0.247868,-1.065926,-0.784694,0.464990,-0.799521,-0.108155
137,-1.118344,-0.175482,-0.784694,-0.706059,-0.799521,-0.515193


In [57]:
MAASTRO_new 

,first_order_Maximum_CT,LBP_201_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,LBP_003_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,1226.795917,0.000026,0.000026,0.000241,0.000026,0.000768
1,1876.570382,0.000167,0.000167,0.000621,0.000056,0.001213
2,1269.217095,0.000000,0.000057,0.000327,0.000286,0.001176
3,1951.924395,0.000080,0.000000,0.001419,0.000160,0.001192
4,1646.853670,0.000035,0.000000,0.000165,0.000071,0.000766
...,...,...,...,...,...,...
94,2311.845731,0.000078,0.000000,0.000739,0.000000,0.001137
95,1299.857157,0.000054,0.000000,0.001130,0.000109,0.000962
96,1256.024949,0.000028,0.000000,0.000226,0.000000,0.000621
97,1668.742962,0.000000,0.000000,0.000296,0.000114,0.000947


In [59]:
MAASTRO_new_std

,first_order_Maximum_CT,LBP_201_PET,LBP_102_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_CT_c16,LBP_003_PET,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,-0.984802,-0.597387,0.033184,-0.811666,-0.238970,-0.960377
1,1.231237,0.946684,1.682719,0.464229,0.263801,0.101598
2,-0.720051,-1.065926,0.693043,-0.453072,1.817599,0.027699
3,1.348682,0.152800,-0.784694,1.579512,1.314752,0.060650
4,0.758335,-0.453142,-0.784694,-1.171859,0.471935,-0.967045
...,...,...,...,...,...,...
94,1.750433,0.126817,-0.784694,0.727222,-0.799521,-0.052777
95,-0.545966,-0.173978,-0.784694,1.322893,0.907497,-0.450240
96,-0.799285,-0.565019,-0.784694,-0.877027,-0.799521,-1.409205
97,0.812692,-1.065926,-0.784694,-0.577241,0.962590,-0.485449


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [61]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 04:30:45,497] A new study created in memory with name: no-name-746e3e9b-43f7-4972-a56e-0bace4227d16


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.6075949367088608


[I 2024-04-19 04:30:47,375] A new study created in memory with name: no-name-864484f1-cac2-4ab5-978f-193cb14662af


Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:30:47,316] Trial 0 finished with value: 0.6640896071389869 and parameters: {}. Best is trial 0 with value: 0.6640896071389869.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6640896071389869], datetime_start=datetime.datetime(2024, 4, 19, 4, 30, 45, 649878), datetime_complete=datetime.datetime(2024, 4, 19, 4, 30, 47, 313302), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6640896071389869


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.19675872550942683
Fold 2 IBS: 0.1783326995533363
Fold 3 IBS: 0.18580016536387306
Fold 4 IBS: 0.2148920002265226
Fold 5 IBS: 0.23991728125827994
[I 2024-04-19 04:30:49,605] Trial 0 finished with value: 0.20314017438228776 and parameters: {}. Best is trial 0 with value: 0.20314017438228776.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20314017438228776], datetime_start=datetime.datetime(2024, 4, 19, 4, 30, 47, 552936), datetime_complete=datetime.datetime(2024, 4, 19, 4, 30, 49, 604428), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20314017438228776


In [63]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [65]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.666
train_ibs:  0.203


#### Test

In [67]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.586
IBS score: 0.239


In [71]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [72]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [73]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 04:31:18,420] A new study created in memory with name: no-name-354004f3-c443-49b5-8d47-a3d9be87b4b9


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5952380952380952
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.7009803921568627
Fold 4 C-index: 0.70042194092827


[I 2024-04-19 04:31:19,863] A new study created in memory with name: no-name-e774119d-9ae3-4778-855d-95344f42aaef


Fold 5 C-index: 0.647887323943662
[I 2024-04-19 04:31:19,780] Trial 0 finished with value: 0.6753341218819494 and parameters: {}. Best is trial 0 with value: 0.6753341218819494.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6753341218819494], datetime_start=datetime.datetime(2024, 4, 19, 4, 31, 18, 627377), datetime_complete=datetime.datetime(2024, 4, 19, 4, 31, 19, 779800), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6753341218819494


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651842800988
Fold 2 IBS: 0.2215779102658637
Fold 3 IBS: 0.20453594230284272
Fold 4 IBS: 0.22473803637388992
Fold 5 IBS: 0.2181243145661695
[I 2024-04-19 04:31:21,569] Trial 0 finished with value: 0.21659054438735512 and parameters: {}. Best is trial 0 with value: 0.21659054438735512.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054438735512], datetime_start=datetime.datetime(2024, 4, 19, 4, 31, 20, 269430), datetime_complete=datetime.datetime(2024, 4, 19, 4, 31, 21, 568572), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054438735512


In [74]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [75]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.675
train_ibs:  0.217


#### Test

In [76]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [77]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.616


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [78]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [79]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 04:31:22,406] A new study created in memory with name: no-name-8260b3ae-b65f-4ca9-8487-09da5fba7796


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6075949367088608


[I 2024-04-19 04:31:25,340] A new study created in memory with name: no-name-b482b541-364e-43f4-9730-f7793b8807eb


Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:31:25,294] Trial 0 finished with value: 0.6659628564387068 and parameters: {}. Best is trial 0 with value: 0.6659628564387068.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6659628564387068], datetime_start=datetime.datetime(2024, 4, 19, 4, 31, 22, 586109), datetime_complete=datetime.datetime(2024, 4, 19, 4, 31, 25, 293445), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6659628564387068


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.19563802137250225
Fold 2 IBS: 0.17860209501109262
Fold 3 IBS: 0.18572177672598686
Fold 4 IBS: 0.2142665104146998
Fold 5 IBS: 0.23919274264809653
[I 2024-04-19 04:31:28,833] Trial 0 finished with value: 0.20268422923447563 and parameters: {}. Best is trial 0 with value: 0.20268422923447563.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.20268422923447563], datetime_start=datetime.datetime(2024, 4, 19, 4, 31, 25, 410467), datetime_complete=datetime.datetime(2024, 4, 19, 4, 31, 28, 832496), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.20268422923447563


In [80]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [81]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.666
train_ibs:  0.203


#### Test

In [82]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [83]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.588


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.238


In [84]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [85]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 04:31:30,206] A new study created in memory with name: no-name-af1496be-e53a-4dfd-8e5d-a2aaf34d6dea


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:31:32,658] Trial 0 finished with value: 0.6650699992958496 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6650699992958496.
Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:31:35,391] Trial 1 finished with value: 0.6650699992958496 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6650699992958496.
Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:31:38,446] Trial 2 finished with value: 0.6650699992958496 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:32:36,160] Trial 24 finished with value: 0.6650699992958496 and parameters: {'l1_ratio': 0.7805647035680947}. Best is trial 6 with value: 0.6659628564387068.
Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:32:39,019] Trial 25 finished with value: 0.6659628564387068 and parameters: {'l1_ratio': 0.9368557715647121}. Best is trial 6 with value: 0.6659628564387068.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.7107843137254902
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6572769953051644
[I 2024-04-19 04:32:40,179] Trial 26 finished with value: 0.6805205170529939 and parameters: {'l1_ratio': 0.01542375755129554

Fold 5 C-index: 0.6572769953051644
[I 2024-04-19 04:33:34,151] Trial 47 finished with value: 0.6805205170529939 and parameters: {'l1_ratio': 0.01366650336233158}. Best is trial 26 with value: 0.6805205170529939.
Fold 1 C-index: 0.6147186147186147
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:33:36,961] Trial 48 finished with value: 0.6642041984300489 and parameters: {'l1_ratio': 0.08891191758251386}. Best is trial 26 with value: 0.6805205170529939.
Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.6862745098039216
Fold 4 C-index: 0.6075949367088608
Fold 5 C-index: 0.6713615023474179
[I 2024-04-19 04:33:39,555] Trial 49 finished with value: 0.6650699992958496 and parameters: {'l1_ratio': 0.18318195239603682}. Best is trial 26 with value: 0.6805205170529939.
Fold 1 C-index: 0.6190476190476191
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index

...

Fold 1 IBS: 0.19567677986005333
Fold 2 IBS: 0.17857093604269592
Fold 3 IBS: 0.18572718284313966
Fold 4 IBS: 0.21425221710181686
Fold 5 IBS: 0.239091161099382
[I 2024-04-19 04:35:33,729] Trial 0 finished with value: 0.20266365538941758 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.20266365538941758.
Fold 1 IBS: 0.1959198487054693
Fold 2 IBS: 0.17847730434007683
Fold 3 IBS: 0.1857458148783586
Fold 4 IBS: 0.2142847137128549
Fold 5 IBS: 0.23874367666989163
[I 2024-04-19 04:35:36,529] Trial 1 finished with value: 0.20263427166133025 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.20263427166133025.
Fold 1 IBS: 0.19600075027660835
Fold 2 IBS: 0.1784456415189215
Fold 3 IBS: 0.18575148926036986
Fold 4 IBS: 0.21425593723877995
Fold 5 IBS: 0.23870102337363563
[I 2024-04-19 04:35:39,389] Trial 2 finished with value: 0.20263096833366304 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.202630968333663

Fold 1 IBS: 0.19602833763428143
Fold 2 IBS: 0.17843162840626733
Fold 3 IBS: 0.18575424861725504
Fold 4 IBS: 0.21425998252982092
Fold 5 IBS: 0.23865919831968643
[I 2024-04-19 04:36:34,187] Trial 25 finished with value: 0.20262667910146223 and parameters: {'l1_ratio': 0.1987244050980175}. Best is trial 22 with value: 0.2026032892471298.
Fold 1 IBS: 0.19583018440273753
Fold 2 IBS: 0.17849747496561694
Fold 3 IBS: 0.1857416331490958
Fold 4 IBS: 0.21424207986970434
Fold 5 IBS: 0.2388567915020896
[I 2024-04-19 04:36:36,188] Trial 26 finished with value: 0.20263363277784885 and parameters: {'l1_ratio': 0.3527001258697712}. Best is trial 22 with value: 0.2026032892471298.
Fold 1 IBS: 0.19571965174961367
Fold 2 IBS: 0.1785447131755078
Fold 3 IBS: 0.18573182518062856
Fold 4 IBS: 0.21426135998165247
Fold 5 IBS: 0.23901391113707188
[I 2024-04-19 04:36:38,261] Trial 27 finished with value: 0.2026542922448949 and parameters: {'l1_ratio': 0.5681788491232276}. Best is trial 22 with value: 0.20260328924

Fold 5 IBS: 0.21809560596130464
[I 2024-04-19 04:37:29,837] Trial 49 finished with value: 0.21641076048670577 and parameters: {'l1_ratio': 0.004297293952703132}. Best is trial 33 with value: 0.20259866106402408.
Fold 1 IBS: 0.1961487812916338
Fold 2 IBS: 0.17835618651303217
Fold 3 IBS: 0.185770146445766
Fold 4 IBS: 0.2142834886793201
Fold 5 IBS: 0.23841899924858256
[I 2024-04-19 04:37:32,241] Trial 50 finished with value: 0.20259552043566692 and parameters: {'l1_ratio': 0.07022059668984147}. Best is trial 50 with value: 0.20259552043566692.
Fold 1 IBS: 0.1961808477488711
Fold 2 IBS: 0.17834591096200678
Fold 3 IBS: 0.1857723338485543
Fold 4 IBS: 0.2142531432163535
Fold 5 IBS: 0.23844492471507603
[I 2024-04-19 04:37:35,555] Trial 51 finished with value: 0.20259943209817233 and parameters: {'l1_ratio': 0.059283402019761226}. Best is trial 50 with value: 0.20259552043566692.
Fold 1 IBS: 0.1961036555809237
Fold 2 IBS: 0.17838532729073253
Fold 3 IBS: 0.18576434866885227
Fold 4 IBS: 0.2142712

Fold 1 IBS: 0.21303488873832213
Fold 2 IBS: 0.2197833194180575
Fold 3 IBS: 0.18577800593309995
Fold 4 IBS: 0.22343747424433247
Fold 5 IBS: 0.23845332542974051
[I 2024-04-19 04:38:32,925] Trial 74 finished with value: 0.21609740275271055 and parameters: {'l1_ratio': 0.028195299315553535}. Best is trial 50 with value: 0.20259552043566692.
Fold 1 IBS: 0.1961845438669615
Fold 2 IBS: 0.17835563899983484
Fold 3 IBS: 0.18577061249803245
Fold 4 IBS: 0.21426541160545684
Fold 5 IBS: 0.23849764786613312
[I 2024-04-19 04:38:35,466] Trial 75 finished with value: 0.20261477096728372 and parameters: {'l1_ratio': 0.06770838940630236}. Best is trial 50 with value: 0.20259552043566692.
Fold 1 IBS: 0.19603171762469188
Fold 2 IBS: 0.178443234622852
Fold 3 IBS: 0.18575415004487017
Fold 4 IBS: 0.21426400153471836
Fold 5 IBS: 0.23867078554393042
[I 2024-04-19 04:38:38,013] Trial 76 finished with value: 0.20263277787421258 and parameters: {'l1_ratio': 0.20072860823217953}. Best is trial 50 with value: 0.20259

Fold 1 IBS: 0.19583855394777175
Fold 2 IBS: 0.1784990783463084
Fold 3 IBS: 0.18574079905797627
Fold 4 IBS: 0.21426788636210636
Fold 5 IBS: 0.2388401592257628
[I 2024-04-19 04:39:38,931] Trial 99 finished with value: 0.20263729538798508 and parameters: {'l1_ratio': 0.382376673820383}. Best is trial 50 with value: 0.20259552043566692.


* Best trial for IBS: 
 FrozenTrial(number=50, state=TrialState.COMPLETE, values=[0.20259552043566692], datetime_start=datetime.datetime(2024, 4, 19, 4, 37, 29, 844731), datetime_complete=datetime.datetime(2024, 4, 19, 4, 37, 32, 240769), params={'l1_ratio': 0.07022059668984147}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=50, value=None)


* Best Score for IBS: 
 0.20259552043566692


In [86]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [87]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.681
train_ibs:  0.203


#### Test

In [88]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [89]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.015423757551295547)

test_cindex : 0.617


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.07022059668984147)

test_ibs:  0.237


In [90]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [91]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 04:39:39,938] A new study created in memory with name: no-name-d064a47e-3a48-4462-8669-542b2bb5a0b3


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.7232142857142857
Fold 3 C-index: 0.6323529411764706
Fold 4 C-index: 0.6139240506329114
Fold 5 C-index: 0.6948356807511737
[I 2024-04-19 04:39:57,723] Trial 0 finished with value: 0.651480110269687 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.651480110269687.
Fold 1 C-index: 0.6406926406926406
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.6078431372549019
Fold 4 C-index: 0.6118143459915611
Fold 5 C-index: 0.676056338028169
[I 2024-04-19 04:40:11,498] Trial 1 finished with value: 0.6554955781077403 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'ma

Fold 5 C-index: 0.6854460093896714
[I 2024-04-19 04:42:42,358] Trial 15 finished with value: 0.6905576634858226 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 383, 'oob_score': True, 'max_samples': 0.4747428535255964, 'max_features': None, 'min_weight_fraction_leaf': 0.19147439464820804, 'warm_start': True}. Best is trial 14 with value: 0.7004823677392317.
Fold 1 C-index: 0.5995670995670995
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.6813725490196079
Fold 4 C-index: 0.7468354430379747
Fold 5 C-index: 0.6948356807511737
[I 2024-04-19 04:42:55,662] Trial 16 finished with value: 0.7052364401894569 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 1, 'n_estimators': 498, 'oob_score': True, 'max_samples': 0.8353789630965257, 'max_features': None, 'min_weight_fraction_leaf': 0.18739880110892437, 'warm_start': True}. Best is trial 16 with value: 0.705236440189456

Fold 1 C-index: 0.6103896103896104
Fold 2 C-index: 0.8660714285714286
Fold 3 C-index: 0.6470588235294118
Fold 4 C-index: 0.7383966244725738
Fold 5 C-index: 0.7089201877934272
[I 2024-04-19 04:45:21,502] Trial 30 finished with value: 0.7141673349512903 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 3, 'n_estimators': 325, 'oob_score': True, 'max_samples': 0.7229708697769008, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.151583696008541, 'warm_start': True}. Best is trial 30 with value: 0.7141673349512903.
Fold 1 C-index: 0.5930735930735931
Fold 2 C-index: 0.8705357142857143
Fold 3 C-index: 0.6568627450980392
Fold 4 C-index: 0.7341772151898734
Fold 5 C-index: 0.7089201877934272
[I 2024-04-19 04:45:29,671] Trial 31 finished with value: 0.7127138910881294 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 3, 'n_estimators': 326, 'oob_score': True, 'max_samples': 0.7487289552088571,

...

Fold 5 IBS: 0.21026303080320738
[I 2024-04-19 04:56:17,716] Trial 31 finished with value: 0.19947895048662703 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 248, 'oob_score': True, 'max_samples': 0.9640858485797439, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.07919805807535037}. Best is trial 15 with value: 0.19786180218775584.
Fold 1 IBS: 0.19629489252012824
Fold 2 IBS: 0.19759768258837943
Fold 3 IBS: 0.18480416566240168
Fold 4 IBS: 0.21321890670626756
Fold 5 IBS: 0.2135292151651214
[I 2024-04-19 04:56:32,638] Trial 32 finished with value: 0.20108897252845964 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 5, 'max_depth': 11, 'n_estimators': 244, 'oob_score': True, 'max_samples': 0.9502203520505428, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.022621037970528283}. Best is trial 15 with value: 0.19786180218775584.
Fold 1 IBS: 0.20008218370007697
Fold 2 IBS: 0.1

Fold 1 IBS: 0.21398232464651643
Fold 2 IBS: 0.22130003244265287
Fold 3 IBS: 0.20480819176804077
Fold 4 IBS: 0.22460990433148936
Fold 5 IBS: 0.21859907545237395
[I 2024-04-19 05:00:49,652] Trial 47 finished with value: 0.21665990572821467 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 11, 'n_estimators': 444, 'oob_score': True, 'max_samples': 0.8235549211572327, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.4234817047062571}. Best is trial 15 with value: 0.19786180218775584.
Fold 1 IBS: 0.20724380394603853
Fold 2 IBS: 0.1964488483468546
Fold 3 IBS: 0.19697222605093012
Fold 4 IBS: 0.2161799369468603
Fold 5 IBS: 0.2109509933041631
[I 2024-04-19 05:01:01,766] Trial 48 finished with value: 0.20555916171896932 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 20, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 268, 'oob_score': False, 'max_samples': 0.5606440507914384, 'max_features': 'log2', 'min_weight_fraction_le

Fold 5 IBS: 0.20741974855549727
[I 2024-04-19 05:05:01,910] Trial 62 finished with value: 0.19647080498147923 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 351, 'oob_score': True, 'max_samples': 0.8590719713823661, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.002132335828788339}. Best is trial 53 with value: 0.19532586703757782.
Fold 1 IBS: 0.1945971546402581
Fold 2 IBS: 0.19921212051462786
Fold 3 IBS: 0.19069889225062295
Fold 4 IBS: 0.2095358322545664
Fold 5 IBS: 0.21153899502669316
[I 2024-04-19 05:05:18,540] Trial 63 finished with value: 0.2011165989373537 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 1, 'max_depth': 18, 'n_estimators': 313, 'oob_score': True, 'max_samples': 0.9011866179753103, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.06358250218509034}. Best is trial 53 with value: 0.19532586703757782.
Fold 1 IBS: 0.19748461096657727
Fold 2 IBS: 0.2008

Fold 1 IBS: 0.1989053774529102
Fold 2 IBS: 0.19827714229698426
Fold 3 IBS: 0.19062588382836543
Fold 4 IBS: 0.2093318900607619
Fold 5 IBS: 0.20997416546820158
[I 2024-04-19 05:09:35,081] Trial 78 finished with value: 0.20142289182144468 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 5, 'max_depth': 17, 'n_estimators': 397, 'oob_score': True, 'max_samples': 0.9323713852052357, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.04247308366617032}. Best is trial 74 with value: 0.1947921479665033.
Fold 1 IBS: 0.18598650836541217
Fold 2 IBS: 0.1884315468040781
Fold 3 IBS: 0.19522799169986055
Fold 4 IBS: 0.22724133183991238
Fold 5 IBS: 0.21327248450930764
[I 2024-04-19 05:09:37,268] Trial 79 finished with value: 0.20203197264371414 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 25, 'oob_score': True, 'max_samples': 0.9993252499502729, 'max_features': 'log2', 'min_weight_fraction_lea

Fold 1 IBS: 0.1968798726882478
Fold 2 IBS: 0.1998433173802242
Fold 3 IBS: 0.18609682967348268
Fold 4 IBS: 0.21023083237520823
Fold 5 IBS: 0.21164269021564036
[I 2024-04-19 05:14:14,519] Trial 94 finished with value: 0.20093870846656064 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.9158810703611461, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.06111476813323878}. Best is trial 74 with value: 0.1947921479665033.
Fold 1 IBS: 0.19370745975298193
Fold 2 IBS: 0.19737884950176132
Fold 3 IBS: 0.1830704811594634
Fold 4 IBS: 0.20837912758814287
Fold 5 IBS: 0.21257524928247132
[I 2024-04-19 05:14:36,876] Trial 95 finished with value: 0.19902223345696415 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 412, 'oob_score': True, 'max_samples': 0.9420898595431166, 'max_features': 'log2', 'min_weight_fraction_leaf

In [92]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [93]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.863
train_ibs:  0.195


#### Test

In [94]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [95]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=19, max_features=None, max_leaf_nodes=12,
                     max_samples=0.630961889364407, min_samples_split=5,
                     min_weight_fraction_leaf=0.011342387005902448,
                     n_estimators=218, random_state=123, warm_start=True)

test_cindex:  0.621


RandomSurvivalForest(max_depth=18, max_features='log2', max_leaf_nodes=17,
                     max_samples=0.8407950651939462, min_samples_leaf=2,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.018820068972654837,
                     n_estimators=68, oob_score=True, random_state=123)

test_ibs:  0.22


In [96]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [97]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [98]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 05:15:50,388] A new study created in memory with name: no-name-4e8544af-61c9-4bdd-a800-c353c9697a4d


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.645021645021645
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.6470588235294118
Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.676056338028169
[I 2024-04-19 05:15:53,769] Trial 0 finished with value: 0.695318892358642 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.695318892358642.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 05:16:02,089] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Bes

Fold 3 C-index: 0.5931372549019608
Fold 4 C-index: 0.6434599156118144
Fold 5 C-index: 0.6338028169014085
[I 2024-04-19 05:17:38,473] Trial 15 finished with value: 0.6682628979159372 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7316118600068109.
Fold 1 C-index: 0.6558441558441559
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.5833333333333334
Fold 4 C-index: 0.6118143459915611
Fold 5 C-index: 0.6455399061032864
[I 2024-04-19 05:17:43,950] Trial 16 finished with value: 0.6626992053973245 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction

Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.6519607843137255
Fold 4 C-index: 0.6877637130801688
Fold 5 C-index: 0.6572769953051644
[I 2024-04-19 05:19:01,625] Trial 30 finished with value: 0.6874414240809373 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 8, 'min_samples_leaf': 9, 'max_depth': 11, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.04228849156393042}. Best is trial 23 with value: 0.7368025709935508.
Fold 1 C-index: 0.6493506493506493
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.6764705882352942
Fold 4 C-index: 0.7130801687763713
Fold 5 C-index: 0.704225352112676
[I 2024-04-19 05:19:05,547] Trial 31 finished with value: 0.7093396374092839 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 5, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 401, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', '

...

Fold 1 IBS: 0.20644252348164513
Fold 2 IBS: 0.2019403019456175
Fold 3 IBS: 0.19518482770678974
Fold 4 IBS: 0.21117332821307674
Fold 5 IBS: 0.21254756150696177
[I 2024-04-19 05:34:14,875] Trial 30 finished with value: 0.2054577085708182 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 10, 'min_samples_leaf': 4, 'max_depth': 5, 'n_estimators': 398, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6204655821432031, 'min_weight_fraction_leaf': 0.08941926090357169}. Best is trial 30 with value: 0.2054577085708182.
Fold 1 IBS: 0.20219437207626473
Fold 2 IBS: 0.20791758297727755
Fold 3 IBS: 0.19810920318501377
Fold 4 IBS: 0.21211014051688712
Fold 5 IBS: 0.21592391251551635
[I 2024-04-19 05:34:29,392] Trial 31 finished with value: 0.20725104225419191 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 10, 'min_samples_leaf': 4, 'max_depth': 1, 'n_estimators': 398, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.62108

Fold 1 IBS: 0.2061060330108144
Fold 2 IBS: 0.20556259199015003
Fold 3 IBS: 0.19681570752543345
Fold 4 IBS: 0.21286266572595886
Fold 5 IBS: 0.21454019901532653
[I 2024-04-19 05:38:05,084] Trial 45 finished with value: 0.20717743945353667 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 8, 'max_depth': 6, 'n_estimators': 420, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.5228042333033645, 'min_weight_fraction_leaf': 0.05423198281417843}. Best is trial 30 with value: 0.2054577085708182.
Fold 1 IBS: 0.20940293520492773
Fold 2 IBS: 0.20804660036517866
Fold 3 IBS: 0.1995328323215525
Fold 4 IBS: 0.2168538036825499
Fold 5 IBS: 0.21460320909795136
[I 2024-04-19 05:38:17,252] Trial 46 finished with value: 0.20968787613443202 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 8, 'max_depth': 5, 'n_estimators': 332, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.365404

Fold 1 IBS: 0.191170709407248
Fold 2 IBS: 0.20034725956824517
Fold 3 IBS: 0.1860917463084432
Fold 4 IBS: 0.2126886619299396
Fold 5 IBS: 0.2135244566742881
[I 2024-04-19 05:40:10,890] Trial 60 finished with value: 0.20076456677763282 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 9, 'n_estimators': 86, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9280982393500873, 'min_weight_fraction_leaf': 0.01669807793304111}. Best is trial 60 with value: 0.20076456677763282.
Fold 1 IBS: 0.1926004831958468
Fold 2 IBS: 0.1987117406768836
Fold 3 IBS: 0.1856472650351245
Fold 4 IBS: 0.2125202423023172
Fold 5 IBS: 0.21157082059492746
[I 2024-04-19 05:40:15,461] Trial 61 finished with value: 0.20021011036101988 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 9, 'n_estimators': 89, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9355045

Fold 1 IBS: 0.2015619084730176
Fold 2 IBS: 0.20340538458456545
Fold 3 IBS: 0.1947998336835993
Fold 4 IBS: 0.21288774116744222
Fold 5 IBS: 0.2122749350143774
[I 2024-04-19 05:41:16,390] Trial 75 finished with value: 0.20498596058460042 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 95, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.855914969192737, 'min_weight_fraction_leaf': 0.03770339568279605}. Best is trial 62 with value: 0.19985113127010729.
Fold 1 IBS: 0.19197793012924233
Fold 2 IBS: 0.21261684937986214
Fold 3 IBS: 0.19593075143531394
Fold 4 IBS: 0.21235581098598563
Fold 5 IBS: 0.21671379378403846
[I 2024-04-19 05:41:18,494] Trial 76 finished with value: 0.2059190271428885 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 28, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.78

Fold 1 IBS: 0.18917643633846162
Fold 2 IBS: 0.21015262595618928
Fold 3 IBS: 0.1935077870422502
Fold 4 IBS: 0.2164703849597155
Fold 5 IBS: 0.2081976082003696
[I 2024-04-19 05:42:18,976] Trial 90 finished with value: 0.20350096849939722 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 21, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9990067443357235, 'min_weight_fraction_leaf': 0.009830617135283846}. Best is trial 62 with value: 0.19985113127010729.
Fold 1 IBS: 0.19666080291709018
Fold 2 IBS: 0.2052297665080179
Fold 3 IBS: 0.19192574041702756
Fold 4 IBS: 0.21136412922820036
Fold 5 IBS: 0.21257426066349058
[I 2024-04-19 05:42:25,547] Trial 91 finished with value: 0.2035509399467653 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 10, 'n_estimators': 131, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0

In [99]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [100]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.808
train_ibs:  0.198


#### Test

In [101]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [102]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=20, max_features=None, max_leaf_nodes=15,
                   max_samples=0.6196177319145886, min_samples_leaf=2,
                   min_samples_split=9,
                   min_weight_fraction_leaf=0.010458229178985291,
                   n_estimators=370, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.614


ExtraSurvivalTrees(max_depth=19, max_features='auto', max_leaf_nodes=20,
                   max_samples=0.9523317599123883, min_samples_leaf=2,
                   min_samples_split=8,
                   min_weight_fraction_leaf=0.0028784190622166937,
                   n_estimators=234, oob_score=True, random_state=123)

IBS: 0.211


In [103]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis


#### Train

In [104]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-19 05:43:48,535] A new study created in memory with name: no-name-1fbe30f5-0754-4ea9-96f1-fe50592806ff


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 05:45:03,960] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 05:45:47,026] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 06:02:30,224] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.7784830438231676, 'learning_rate': 0.02439641670004065, 'dropout_rate': 0.4821375662037144, 'n_estimators': 148, 'criterion': 'friedman_mse', 'ccp_alpha': 6.245821408640139, 'min_weight_fraction_leaf': 0.3469021849356369, 'max_features': 1, 'min_impurity_decrease': 0.002901900339595834, 'validation_fraction': 0.40492470595401014, 'min_samples_split': 6, 'max_leaf_nodes': 15, 'min_samples_leaf': 12, 'max_depth': 5}. Best is trial 9 with value: 0.6608490969281925.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 06:04:58,521] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.5522283925781899, 'learning_rate': 0.010913227078192221, 'dropout_rate': 0.275468298707492, 'n_estimators': 392, 'criterion': 'squared_error', 'c

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 06:20:09,655] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8806073308147355, 'learning_rate': 0.022376976197093057, 'dropout_rate': 0.6881130985931054, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 2.7801390404743485, 'min_weight_fraction_leaf': 0.14826348238129222, 'max_features': 1, 'min_impurity_decrease': 3.2746646321179855e-06, 'validation_fraction': 0.654336976255422, 'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 14, 'max_depth': 5}. Best is trial 9 with value: 0.6608490969281925.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-19 06:22:19,784] Trial 27 finished with value: 0.5 and parameters: {'subsample': 0.6416570269538786, 'learning_rate': 0.08875752985571464, 'dropout_rate': 0.19627515205758927, 'n_estimators': 344, 'criterion': 'friedman_mse'

...

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-19 08:29:22,863] Trial 45 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.5303064017076159, 'learning_rate': 0.013591809817627606, 'dropout_rate': 0.4506745791740544, 'n_estimators': 401, 'criterion': 'friedman_mse', 'ccp_alpha': 2.5557833119257145, 'min_weight_fraction_leaf': 0.3344828403640176, 'max_features': 0.1, 'min_impurity_decrease': 0.02254931450703544, 'validation_fraction': 0.5042239156018957, 'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 15, 'max_depth': 14}. Best is trial 9 with value: 0.2165031446823389.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 08:30:39,373] Trial 46 finished with value: 0.21659054862241586 and parameters: {'subsam

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 08:38:53,402] Trial 56 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.1173095704794419, 'learning_rate': 0.022797925457682687, 'dropout_rate': 0.2500192595896841, 'n_estimators': 209, 'criterion': 'squared_error', 'ccp_alpha': 1.515994978548666, 'min_weight_fraction_leaf': 0.04557038009296144, 'max_features': 'sqrt', 'min_impurity_decrease': 8.288990661784225e-06, 'validation_fraction': 0.6682715637714014, 'min_samples_split': 13, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 7}. Best is trial 9 with value: 0.2165031446823389.
Fold 1 IBS: 0.21382373751543124
Fold 2 IBS: 0.2214840159838248
Fold 3 IBS: 0.20428815063117983
Fold 4 IBS: 0.22465724629235706
Fold 5 IBS: 0.21804196994519903
[I 2024-04-19 08:39:30,711] Trial 57 finished with value: 0.2164590240735984 and parameters: {'subsample': 0.553868700002313, 'le

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609563
[I 2024-04-19 08:52:35,642] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.44533772449036413, 'learning_rate': 0.0814448088163274, 'dropout_rate': 0.8313520760085876, 'n_estimators': 440, 'criterion': 'friedman_mse', 'ccp_alpha': 0.5496755232313972, 'min_weight_fraction_leaf': 0.23846073397265538, 'max_features': 'log2', 'min_impurity_decrease': 0.0245829172127866, 'validation_fraction': 0.8751349059723512, 'min_samples_split': 17, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 4}. Best is trial 63 with value: 0.21613552181537923.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609574
[I 2024-04-19 08:54:03,012] Trial 68 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.49945879365729917, 

Fold 2 IBS: 0.2207305662643275
Fold 3 IBS: 0.20400135988740564
Fold 4 IBS: 0.22440754040830044
Fold 5 IBS: 0.21783913995162144
[I 2024-04-19 09:05:39,249] Trial 78 finished with value: 0.21609606173190254 and parameters: {'subsample': 0.3352733938121141, 'learning_rate': 0.07411999177509665, 'dropout_rate': 0.8605229004531987, 'n_estimators': 473, 'criterion': 'squared_error', 'ccp_alpha': 0.023234528924181166, 'min_weight_fraction_leaf': 0.20870378808533252, 'max_features': 'log2', 'min_impurity_decrease': 0.007686471270079544, 'validation_fraction': 0.6325076348170467, 'min_samples_split': 19, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 5}. Best is trial 75 with value: 0.21591584315549578.
Fold 1 IBS: 0.2136232633771708
Fold 2 IBS: 0.2208637477666032
Fold 3 IBS: 0.2041050225232073
Fold 4 IBS: 0.22443455729626233
Fold 5 IBS: 0.2178626020978067
[I 2024-04-19 09:06:07,014] Trial 79 finished with value: 0.21617783861221004 and parameters: {'subsample': 0.33166032751272273,

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-19 09:12:40,387] Trial 89 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.40845912170347753, 'learning_rate': 0.07409455257113193, 'dropout_rate': 0.9031580025226588, 'n_estimators': 489, 'criterion': 'squared_error', 'ccp_alpha': 1.426169496743956, 'min_weight_fraction_leaf': 0.17670307432796536, 'max_features': 'log2', 'min_impurity_decrease': 0.001833888378396466, 'validation_fraction': 0.7135149436193794, 'min_samples_split': 20, 'max_leaf_nodes': 13, 'min_samples_leaf': 13, 'max_depth': 3}. Best is trial 75 with value: 0.21591584315549578.
Fold 1 IBS: 0.2139763741496877
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473191517656796
Fold 5 IBS: 0.21812464788704478
[I 2024-04-19 09:13:31,460] Trial 90 finished with value: 0.21658936034403223 and parameters: {'subsample': 0.2452119153654136,

In [105]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [106]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.662
train_ibs:  0.216


#### Test

In [107]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [108]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.00443168890264431,
                                 dropout_rate=0.7939431243365798,
                                 learning_rate=0.09922718569465194,
                                 max_depth=12, max_features='sqrt',
                                 max_leaf_nodes=20,
                                 min_impurity_decrease=8.596034523879625e-05,
                                 min_samples_leaf=11, min_samples_split=5,
                                 min_weight_fraction_leaf=0.34678735520439286,
                                 n_estimators=279, random_state=123,
                                 subsample=0.293918127029324,
                                 validation_fraction=0.3332285029879782)

C-index score: 0.595


GradientBoostingSurvivalAnalysis(ccp_alpha=0.009365616677551508,
                                 dropout_rate=0.7826693566389662,
                                 learning_rate=0.08843409287484169, max_depth=4,
                                 max_features='log2', max_leaf_nodes=14,
                                 min_impurity_decrease=0.0008096076350961149,
                                 min_samples_leaf=16, min_samples_split=19,
                                 min_weight_fraction_leaf=0.2812508412156809,
                                 n_estimators=403, random_state=123,
                                 subsample=0.5200735929114648,
                                 validation_fraction=0.7004324115891782)

IBS: 0.221


In [109]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [110]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [111]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')

            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)

                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-19 09:22:05,815] A new study created in memory with name: no-name-1b0c7f0e-c0a6-4d65-a4f3-1df90e419fb1


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.6244131455399061
[I 2024-04-19 09:22:08,496] Trial 0 finished with value: 0.6830731061043034 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6830731061043034.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.6244131455399061
[I 2024-04-19 09:22:32,009] Trial 1 finished with value: 0.6830731061043034 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6830731061043034.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7156862745098039
Fold 

Fold 1 C-index: 0.6060606060606061
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.7058823529411765
Fold 4 C-index: 0.7130801687763713
Fold 5 C-index: 0.6525821596244131
[I 2024-04-19 09:25:33,436] Trial 19 finished with value: 0.682842486051942 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.5750257507026826, 'n_estimators': 131, 'learning_rate': 0.009453412662102966}. Best is trial 16 with value: 0.6855981769358904.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.7510548523206751
Fold 5 C-index: 0.6244131455399061
[I 2024-04-19 09:25:42,853] Trial 20 finished with value: 0.6850932812629744 and parameters: {'subsample': 0.3813470543085155, 'dropout_rate': 0.2342004098513716, 'n_estimators': 258, 'learning_rate': 0.03971460589320458}. Best is trial 16 with value: 0.6855981769358904.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.7205882352941176


Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7142857142857143
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.759493670886076
Fold 5 C-index: 0.6244131455399061
[I 2024-04-19 09:29:07,411] Trial 38 finished with value: 0.6841024735474831 and parameters: {'subsample': 0.3631428306108071, 'dropout_rate': 0.5184650999753722, 'n_estimators': 289, 'learning_rate': 0.019705698270018484}. Best is trial 16 with value: 0.6855981769358904.
Fold 1 C-index: 0.5974025974025974
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.7107843137254902
Fold 4 C-index: 0.7172995780590717
Fold 5 C-index: 0.6338028169014085
[I 2024-04-19 09:29:16,842] Trial 39 finished with value: 0.6782864326462851 and parameters: {'subsample': 0.21082898345107914, 'dropout_rate': 0.22565165560333753, 'n_estimators': 235, 'learning_rate': 0.0666636052087898}. Best is trial 16 with value: 0.6855981769358904.
Fold 1 C-index: 0.5974025974025974
Fold 2 C-index: 0.7321428571428571
Fold 3 C-index: 0.7205882352941176


...

Fold 2 IBS: 0.19581322731122527
Fold 3 IBS: 0.18012266008596378
Fold 4 IBS: 0.2080662242520272
Fold 5 IBS: 0.228727148642183
[I 2024-04-19 09:42:57,795] Trial 19 finished with value: 0.20223919957345254 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 7 with value: 0.2005497416666168.
Fold 1 IBS: 0.19900268568861765
Fold 2 IBS: 0.19673124531818784
Fold 3 IBS: 0.18097673731541652
Fold 4 IBS: 0.20623674025628727
Fold 5 IBS: 0.22394152158247504
[I 2024-04-19 09:42:59,521] Trial 20 finished with value: 0.20137778603219686 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 7 with value: 0.2005497416666168.
Fold 1 IBS: 0.20102182120527576
Fold 2 IBS: 0.20177601792835187
Fold 3 IBS: 0.1880831686670229
Fold 4 IBS: 0.21737450486532756
Fold 5 IBS: 0.22131729031914832
[I 2024-04-19

Fold 3 IBS: 0.1822935200256659
Fold 4 IBS: 0.2097335127069102
Fold 5 IBS: 0.22465026642731892
[I 2024-04-19 09:44:04,219] Trial 38 finished with value: 0.20236552979418806 and parameters: {'subsample': 0.17017303600682596, 'dropout_rate': 0.6744028301161138, 'n_estimators': 84, 'learning_rate': 0.02105811333204098}. Best is trial 36 with value: 0.20053173745597602.
Fold 1 IBS: 0.2293570713772131
Fold 2 IBS: 0.21090831309405633
Fold 3 IBS: 0.1854937460956396
Fold 4 IBS: 0.2135188317245189
Fold 5 IBS: 0.3310658823973223
[I 2024-04-19 09:44:09,535] Trial 39 finished with value: 0.23406876893775003 and parameters: {'subsample': 0.3439557983328422, 'dropout_rate': 0.844167451106476, 'n_estimators': 194, 'learning_rate': 0.051973238153326454}. Best is trial 36 with value: 0.20053173745597602.
Fold 1 IBS: 0.21206377067603904
Fold 2 IBS: 0.1864274650720211
Fold 3 IBS: 0.17644966324602238
Fold 4 IBS: 0.19294957610953867
Fold 5 IBS: 0.2664281407354722
[I 2024-04-19 09:44:13,281] Trial 40 finishe

Fold 3 IBS: 0.17849049434659467
Fold 4 IBS: 0.20302608352234905
Fold 5 IBS: 0.22884938440987548
[I 2024-04-19 09:45:57,646] Trial 57 finished with value: 0.20081171638871226 and parameters: {'subsample': 0.31114891339062667, 'dropout_rate': 0.6489378315882792, 'n_estimators': 251, 'learning_rate': 0.007801704420672669}. Best is trial 50 with value: 0.2002407140843617.
Fold 1 IBS: 0.2180531959388824
Fold 2 IBS: 0.1850434781595872
Fold 3 IBS: 0.18125157795710164
Fold 4 IBS: 0.1957099312923242
Fold 5 IBS: 0.2808496755196348
[I 2024-04-19 09:46:04,260] Trial 58 finished with value: 0.21218157177350605 and parameters: {'subsample': 0.3221818826721476, 'dropout_rate': 0.5833347466815137, 'n_estimators': 202, 'learning_rate': 0.027679523779857756}. Best is trial 50 with value: 0.2002407140843617.
Fold 1 IBS: 0.21120995027458148
Fold 2 IBS: 0.1838676821243201
Fold 3 IBS: 0.17761827785583964
Fold 4 IBS: 0.1945128293118185
Fold 5 IBS: 0.2635788119779368
[I 2024-04-19 09:46:12,678] Trial 59 finis

Fold 3 IBS: 0.175369518258988
Fold 4 IBS: 0.19886694224374504
Fold 5 IBS: 0.23393332559570487
[I 2024-04-19 09:48:00,944] Trial 76 finished with value: 0.1997871693527889 and parameters: {'subsample': 0.5836047825601226, 'dropout_rate': 0.7414222854721267, 'n_estimators': 134, 'learning_rate': 0.017294952519981824}. Best is trial 76 with value: 0.1997871693527889.
Fold 1 IBS: 0.1990704813975664
Fold 2 IBS: 0.19235978313316857
Fold 3 IBS: 0.1756310426455353
Fold 4 IBS: 0.199178691043083
Fold 5 IBS: 0.23281528440987867
[I 2024-04-19 09:48:04,603] Trial 77 finished with value: 0.19981105652584638 and parameters: {'subsample': 0.610825053306389, 'dropout_rate': 0.7342999581448342, 'n_estimators': 136, 'learning_rate': 0.016329266673119444}. Best is trial 76 with value: 0.1997871693527889.
Fold 1 IBS: 0.20021184069742604
Fold 2 IBS: 0.1898669747346325
Fold 3 IBS: 0.17413074885324495
Fold 4 IBS: 0.19575506187113495
Fold 5 IBS: 0.23873279804958214
[I 2024-04-19 09:48:08,287] Trial 78 finished

Fold 3 IBS: 0.17467023221093744
Fold 4 IBS: 0.19067718405926123
Fold 5 IBS: 0.25649411258858085
[I 2024-04-19 09:49:05,047] Trial 95 finished with value: 0.20312288163277703 and parameters: {'subsample': 0.742801951389467, 'dropout_rate': 0.6950389161933, 'n_estimators': 126, 'learning_rate': 0.029845688472146227}. Best is trial 80 with value: 0.19960439357370313.
Fold 1 IBS: 0.19915942234350345
Fold 2 IBS: 0.19639364745164575
Fold 3 IBS: 0.1785603826338131
Fold 4 IBS: 0.20275925974876494
Fold 5 IBS: 0.22744247015099853
[I 2024-04-19 09:49:07,487] Trial 96 finished with value: 0.20086303646574516 and parameters: {'subsample': 0.6896197002613113, 'dropout_rate': 0.8197700303393364, 'n_estimators': 83, 'learning_rate': 0.021117969427201858}. Best is trial 80 with value: 0.19960439357370313.
Fold 1 IBS: 0.20018341772485804
Fold 2 IBS: 0.1898492293864333
Fold 3 IBS: 0.17423959635583383
Fold 4 IBS: 0.19657311132315583
Fold 5 IBS: 0.23867363903033434
[I 2024-04-19 09:49:10,054] Trial 97 fini

In [112]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [113]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.689
train_ibs:  0.2


#### Test

In [114]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [115]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5421969302450425,
                                              learning_rate=0.0332916598506947,
                                              n_estimators=251,
                                              random_state=123,
                                              subsample=0.20242575932269302)

C-index score: 0.635


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7357326565933753,
                                              learning_rate=0.01759253922056418,
                                              n_estimators=140,
                                              random_state=123,
                                              subsample=0.7315717945946657)

IBS: 0.206


In [116]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [117]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.863,1.0
ExtraSurvivalTrees,0.808,2.0
ComponentwiseGradientBoosting,0.689,3.0
CoxElastic,0.681,4.0
CoxRidge,0.675,5.0
CoxLasso,0.666,6.0
CoxPH,0.664,7.0
GradientBoosting,0.662,8.0


In [118]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.195,1.0
ExtraSurvivalTrees,0.198,2.0
ComponentwiseGradientBoosting,0.200,3.0
CoxPH,0.203,5.0
CoxLasso,0.203,5.0
CoxElastic,0.203,5.0
GradientBoosting,0.216,7.0
CoxRidge,0.217,8.0


In [119]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ComponentwiseGradientBoosting,0.635,1.0
Randomsurvivalforest,0.621,2.0
CoxElastic,0.617,3.0
CoxRidge,0.616,4.0
ExtraSurvivalTrees,0.614,5.0
GradientBoosting,0.595,6.0
CoxLasso,0.588,7.0
CoxPH,0.586,8.0


In [120]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
ComponentwiseGradientBoosting,0.206,1.0
ExtraSurvivalTrees,0.211,2.0
Randomsurvivalforest,0.220,3.0
CoxRidge,0.221,4.5
GradientBoosting,0.221,4.5
CoxElastic,0.237,6.0
CoxLasso,0.238,7.0
CoxPH,0.239,8.0


In [121]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/yeojohnson/plsr/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_yeojohnson_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [122]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-19
